In [1]:
import pandas as pd
from statsmodels.tsa.ardl import ardl_select_order

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

# create policy dummy if not already
df["ron95_policy"] = (df.index >= "2025-09-01").astype(int)

df = df[["ron95", "brent", "usdmyr", "ron95_policy"]]
df = df.asfreq("ME")

df.head()


,ron95,brent,usdmyr,ron95_policy
date,,,,
2020-01-31,2.0800,63.645455,4.080110,0
2020-02-29,2.0680,55.657000,4.159900,0
2020-03-31,1.6325,32.011364,4.298409,0
2020-04-30,1.2625,18.378500,4.352486,0
2020-05-31,1.3240,29.378947,4.340906,0


In [2]:
sel = ardl_select_order(
    endog=df["ron95"],
    exog=df[["brent", "usdmyr", "ron95_policy"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print(sel.model)


In [7]:
ardl_r95 = sel.model.fit()
print(ardl_r95.summary())


                              ARDL Model Results                              
Dep. Variable:                  ron95   No. Observations:                   71
Model:                  ARDL(3, 0, 2)   Log Likelihood                 155.313
Method:               Conditional MLE   S.D. of innovations              0.025
Date:                Thu, 08 Jan 2026   AIC                           -292.625
Time:                        08:54:44   BIC                           -272.649
Sample:                    04-30-2020   HQIC                          -284.710
                         - 11-30-2025                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.2136      0.069      3.087      0.003       0.075       0.352
ron95.L1            1.6873      0.049     34.429      0.000       1.589       1.785
ron95.L2           -1.2617      

In [4]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan


In [9]:
res = ardl_r95
def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}.")

    # align exog to residual sample
    X0 = X0_full[-n_u:, :]

    # auxiliary regression: u_t on X + lagged u
    U_lags = lagmat(u, maxlag=nlags, trim="both")
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Non-finite values (NaN/inf) in BG auxiliary regression inputs.")

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(X0_full.shape[0]),
    }

# IMPORTANT: set res to your fitted ARDL results object
# res = ardl_r95   # <- example, replace with your variable name

bg_out = bg_test_ardl_aligned(res, nlags=2)
bg_out


{'LM stat': 2.0737379648481853,
 'LM p-value': 0.3545630890958924,
 'F stat': 0.9447362516855178,
 'F p-value': 0.39439593600421075,
 'nobs_aux': 66,
 'resid_len': 68,
 'exog_rows_full': 71}

In [10]:
def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]  # align
    X0 = sm.add_constant(X0, has_constant="add")  # force constant

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1]),
    }

bp_out = bp_test_ardl(res)
bp_out


{'LM stat': 20.570030669063275,
 'LM p-value': 0.00012929588586498358,
 'F stat': 9.252110577558112,
 'F p-value': 3.616551343173059e-05,
 'k_exog': 4}

In [11]:
import joblib
joblib.dump(res, "../data/joblib/ardl_ron95_brent.joblib")

['../data/joblib/ardl_ron95_brent.joblib']